# Figure 3 · 全样本的神经—化学对应

**主图采用先前选定的 380 项 reference-based log₂FC 比较**，保持原图的双距离矩阵和右侧配对密度布局。它覆盖同样的 106 株、13 类神经元和 5,565 个唯一菌株对。先展示输入、距离计算、历史核对和主图，再保留 162 项浓度版本及相关探索作为对照。

两套化学表示来自同一份原始测量，但特征选择与处理不同：380 项 FC 继承缺失填零、加 1 和不同参考组归一化；162 项浓度只保留完整正值且通过 QC 的项目，没有填补、加 1 或除参考。**Figure 4 的局部 chemical-state score 继续使用 162 项浓度输入。**

本册不重新处理原始荧光、不拟合新神经模板、不搜索模型或重新拟合 HMDS。主图与对照图分别保存数值、图注和来源，不混用相关系数。

In [ ]:
from pathlib import Path
import sys

# Launch from the repository, poster/, or poster/notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'poster/src/poster_analysis').is_dir() and (p / 'reports').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the bacteria_analysis repository.')
sys.path.insert(0, str(ROOT / 'poster/src'))

import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata
from IPython.display import Image, Markdown, display
from poster_analysis.paths import report_path, figure_dir, table_dir
from poster_analysis.constants import NEURONS, SEED
from poster_analysis.style import apply_style, save_figure
from poster_analysis.global_comparison import (
    distance_matrices, paired_distances, describe_correspondence,
    plot_global_comparison, plot_reference_fc_comparison, plot_within_between,
)

%matplotlib inline
apply_style()
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 8)
CHEM = report_path('chemistry')
DIST = report_path('genus_distances')
RESP = report_path('responses')
PATTERNS = report_path('genus_patterns')
SAVE_FIGURES = True
SHOW_HISTORICAL = True  # Saved supporting views only; no embedding refit.
REFERENCE_CHEM = ROOT / 'reports/population/population_first_20260930'

## 主图 1. 原 380 项化学表征与 13 维神经系数

化学输入为已保存的 `aligned_chemical_log2fc_paired.csv`，其中已经是 log₂FC，**不能再次取 log 或替换为当前浓度**。原 FC 的数值处理包括缺失填零、加 1，并使用样本对应的参考组。该定义延续原全局比较；它不等同于“380 项均通过当前 QC”。

神经输入仍是 individual-SNR ≥ 0.5 的 13 类有符号模板系数。用于余弦距离时，先除以每株向量的 L2 范数不会改变方向。下面对齐全部 106 株，不另挑样本或菌株对。

In [ ]:
main_chemical_log2fc = pd.read_csv(
    REFERENCE_CHEM / 'tables/aligned_chemical_log2fc_paired.csv', index_col=0)
main_coefficients = pd.read_csv(RESP / 'tables/strain_coefficients.csv', index_col=0).loc[:, NEURONS]
main_ids = main_coefficients.index
main_chemical_log2fc = main_chemical_log2fc.loc[main_ids]
assert main_chemical_log2fc.shape == (106, 380) and main_coefficients.shape == (106, 13)
assert main_ids.is_unique and main_chemical_log2fc.columns.is_unique
assert np.isfinite(main_chemical_log2fc.to_numpy()).all()
assert np.isfinite(main_coefficients.to_numpy()).all()
main_neural_unit = main_coefficients.div(np.linalg.norm(main_coefficients, axis=1), axis=0)
np.testing.assert_allclose(np.linalg.norm(main_neural_unit, axis=1), 1, atol=1e-12)
display(pd.DataFrame({'input': ['Reference-based chemical log2FC', 'Neural signed coefficients'],
                     'strains': [106, 106], 'dimensions': [380, 13]}))

## 主图 2. 一对菌株的距离 → 全部配对 → 历史核对

化学距离为 380 个 log₂FC 差值的 RMS；神经距离为 13 维系数的 `1−cosine`。两者单位不同，绝对数值不能互相比大小。先展开 A001/A002 的计算，再计算全矩阵。

最后与旧图使用的 `rdm_chemical.csv` 和 `rdm_filtered.csv` 核对。绘图采用核对后的原矩阵，因此不因 CSV 重算的舍入改变原图分箱。

In [ ]:
MAIN_STRAIN_A, MAIN_STRAIN_B = 'A001', 'A002'
main_delta = (main_chemical_log2fc.loc[MAIN_STRAIN_A] - main_chemical_log2fc.loc[MAIN_STRAIN_B]).to_numpy()
main_u = main_neural_unit.loc[MAIN_STRAIN_A].to_numpy()
main_v = main_neural_unit.loc[MAIN_STRAIN_B].to_numpy()
display(pd.Series({
    'Chemical RMS difference across 380 log2FC annotations': np.sqrt(np.mean(main_delta ** 2)),
    'Neural cosine distance': 1 - np.dot(main_u, main_v) / (np.linalg.norm(main_u) * np.linalg.norm(main_v)),
}, name=f'{MAIN_STRAIN_A} vs {MAIN_STRAIN_B}'))

main_chemical_computed = squareform(pdist(main_chemical_log2fc.to_numpy(), metric='euclidean') / np.sqrt(380))
main_neural_computed = np.clip(squareform(pdist(main_neural_unit.to_numpy(), metric='cosine')), 0, 2)
main_chemical_distance = pd.read_csv(RESP / 'tables/rdm_chemical.csv', index_col=0).loc[main_ids, main_ids]
main_neural_distance = pd.read_csv(RESP / 'tables/rdm_filtered.csv', index_col=0).loc[main_ids, main_ids]
np.testing.assert_allclose(main_chemical_computed, main_chemical_distance, atol=1e-12, rtol=1e-12)
np.testing.assert_allclose(main_neural_computed, main_neural_distance, atol=1e-12, rtol=1e-12)
display(pd.Series({
    'Maximum chemical difference from original': np.max(np.abs(main_chemical_computed - main_chemical_distance.to_numpy())),
    'Maximum neural difference from original': np.max(np.abs(main_neural_computed - main_neural_distance.to_numpy())),
}))

In [ ]:
main_i, main_j = np.triu_indices(len(main_ids), 1)
main_pairs = pd.DataFrame({
    'strain_a': main_ids[main_i], 'strain_b': main_ids[main_j],
    'chemical_distance': main_chemical_distance.to_numpy()[main_i, main_j],
    'neural_distance': main_neural_distance.to_numpy()[main_i, main_j],
})
assert len(main_pairs) == 5565 and not main_pairs[['strain_a', 'strain_b']].duplicated().any()
main_metrics = describe_correspondence(main_pairs)
main_r = np.corrcoef(main_pairs.chemical_distance, main_pairs.neural_distance)[0, 1]
main_rho = np.corrcoef(rankdata(main_pairs.chemical_distance), rankdata(main_pairs.neural_distance))[0, 1]
np.testing.assert_allclose([main_r, main_rho], [main_metrics['pearson_r'], main_metrics['spearman_rho']])
old_summary = pd.read_csv(RESP / 'tables/rdm_comparison_summary.csv')
old_metric = old_summary.loc[(old_summary.comparison == 'filtered_vs_chemical') &
                             (old_summary.support == 'own_valid_pairs')].iloc[0]
np.testing.assert_allclose([main_r, main_rho], old_metric[['pearson', 'spearman']].to_numpy(float), atol=1e-12)
display(pd.Series(main_metrics, name='Main: 380 reference-based log2FC annotations'))

## 主图 3. 保持原全局比较图

保持原图的数值、样本顺序和绘图定义：神经色阶 0–2，化学色阶 0–观测最大值，右侧 `RdBu_r` 密度色图、27 格。右侧每个六边形统计同一对菌株的化学距离和神经距离，红色表示较多菌株对。它展示较弱的正向关联和较大的分散，不表示确定的线性映射。

5,565 个菌株对共享 106 株细菌，不能作为 5,565 个独立生物学重复；这里只报告描述性相关。

In [ ]:
main_order = pd.read_csv(RESP / 'tables/figure_row_order.csv').sort_values('position').strain.tolist()
assert set(main_order) == set(main_ids) and len(main_order) == len(main_ids)
fig = plot_reference_fc_comparison(main_chemical_distance, main_neural_distance, order=main_order)
if SAVE_FIGURES:
    save_figure(fig, figure_dir('main') / 'fig03_global_chemical_neural', dpi=180)
plt.show()

main_pairs.to_csv(table_dir() / 'fig03_matched_pairs.csv', index=False)
pd.Series(main_metrics).to_json(table_dir() / 'fig03_descriptive_correspondence.json', indent=2)
main_display_parameters = {
    'role': 'main_380_reference_log2fc', 'n_chemical_annotations': 380,
    'original_figure': str((RESP / 'figures/03_neural_chemical_rdm.png').relative_to(ROOT)),
    'neural_heatmap_limits': [0, 2],
    'chemical_heatmap_limits': [0, float(main_chemical_distance.max().max())],
    'neural_heatmap_normalization': 'linear', 'pair_cmap': 'RdBu_r', 'pair_gridsize': 27,
    'pair_plot_y_limits': [0, 2], 'n_unique_pairs': len(main_pairs),
    'distance_values_modified': False,
    'retained_162_figure': 'poster/figures/supporting/fig03_global_chemical_neural_162concentration.png',
}
(table_dir() / 'fig03_display_parameters.json').write_text(json.dumps(main_display_parameters, indent=2) + '\n')
main_source_paths = [REFERENCE_CHEM / 'tables/aligned_chemical_log2fc_paired.csv']
main_source_paths += [RESP / 'tables' / name for name in [
    'strain_coefficients.csv', 'rdm_chemical.csv', 'rdm_filtered.csv',
    'figure_row_order.csv', 'rdm_comparison_summary.csv']]
main_source_paths += [ROOT / 'poster/src/poster_analysis/global_comparison.py']
main_sources = pd.DataFrame([{'path': str(path.relative_to(ROOT)),
    'sha256': hashlib.sha256(path.read_bytes()).hexdigest()} for path in main_source_paths])
main_sources.to_csv(table_dir() / 'fig03_sources.csv', index=False)

---

以下保留 **162 项浓度对照及其探索**，图导出到 `figures/supporting/fig03_global_chemical_neural_162concentration`。上面 380 项主图的导出不会被后续单元覆盖。

## 对照 1. 162 项浓度输入

以下保留整理时生成的 162 项对照版本，它与 Figure 4 使用同一套浓度输入。每项等权，**不逐项除以跨菌株 SD，也不按 family 重加权**；Figure 4 的局部 score 则先在属内标准化、聚类和 family 等权。

这里的主图仍是上面的 380 项 FC 版本。本段及后续属内/属间探索属于明确标注的 162 项对照，不用于替换主图的距离、坐标或相关系数。

神经输入是同样 13 类系数的 L2 单位向量，下面会与主图神经距离核对。

In [ ]:
chemical_log2 = pd.read_csv(CHEM / 'tables/fresh_chemical_log2.csv', index_col='strain')
neural_unit = pd.read_csv(CHEM / 'tables/neural_unit_coefficients.csv', index_col='strain')
context = pd.read_csv(CHEM / 'tables/sample_context.csv', index_col='strain')
features = pd.read_csv(CHEM / 'tables/fresh_feature_metadata.csv', index_col='metabolite')
assert chemical_log2.shape == (106, 162) and neural_unit.shape == (106, 13)
assert set(chemical_log2.index) == set(neural_unit.index) == set(context.index)
chemical_log2 = chemical_log2.loc[neural_unit.index]
context = context.loc[neural_unit.index]
assert list(neural_unit.columns) == NEURONS
assert np.isfinite(chemical_log2.to_numpy()).all()
assert np.isfinite(neural_unit.to_numpy()).all()
assert features.qc_rsd_recomputed.le(.30).all()
np.testing.assert_allclose(np.linalg.norm(neural_unit, axis=1), 1, atol=1e-12)
pd.DataFrame({'Input': ['Chemical log2 concentration', 'Unit neural coefficients'],
              'Strains': [len(chemical_log2), len(neural_unit)],
              'Dimensions': [chemical_log2.shape[1], neural_unit.shape[1]]})

## 对照 2. 先手算一个菌株对

化学距离是两个 log 浓度向量差的均方根：

\[d_C(i,k)=\sqrt{\frac{1}{162}\sum_j(x_{ij}-x_{kj})^2}.\]

神经距离是 13 维响应的 cosine distance：

\[d_N(i,k)=1-\frac{u_i^T u_k}{\|u_i\|\|u_k\|}.\]

下面两个可替换的菌株只是计算示例，没有参与筛选整张图的样本。化学距离和神经距离的单位不同，绝对数值不能互相比大小。

In [ ]:
STRAIN_A, STRAIN_B = 'A001', 'A002'
log_difference = chemical_log2.loc[STRAIN_A] - chemical_log2.loc[STRAIN_B]
u, v = neural_unit.loc[STRAIN_A].to_numpy(), neural_unit.loc[STRAIN_B].to_numpy()
example_chemical = np.sqrt(np.mean(log_difference.to_numpy() ** 2))
example_neural = 1 - np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
pd.Series({'Chemical RMS log2 difference': example_chemical,
           'Neural cosine distance': example_neural}, name=f'{STRAIN_A} vs {STRAIN_B}')

In [ ]:
# Inspect both measurements for exactly the same illustrative pair.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), layout='constrained')
axes[0].scatter(chemical_log2.loc[STRAIN_A], chemical_log2.loc[STRAIN_B], s=12, alpha=.65)
lim = [float(chemical_log2.loc[[STRAIN_A,STRAIN_B]].min().min()),
       float(chemical_log2.loc[[STRAIN_A,STRAIN_B]].max().max())]
axes[0].plot(lim, lim, color='gray', lw=.8)
axes[0].set(xlabel=f'{STRAIN_A}: log2 concentration', ylabel=f'{STRAIN_B}: log2 concentration',
            title='All 162 annotations')
for aid in [STRAIN_A, STRAIN_B]:
    axes[1].plot(NEURONS, neural_unit.loc[aid], marker='o', markersize=3, label=aid)
axes[1].axhline(0, color='gray', lw=.6)
axes[1].tick_params(axis='x', labelrotation=60)
axes[1].set(ylabel='Unit coefficient', title='All 13 neurons')
axes[1].legend(frameon=False)
if SAVE_FIGURES:
    save_figure(fig, figure_dir('supporting') / 'fig03_pair_calculation_example')
plt.show()

## 对照 3. 对全部 106 株计算，并核对既有结果

`pdist` 只计算不同菌株的无序配对；这里有 \(106\times105/2=5565\) 对。下面保留核心两行计算；可复用函数用于独立核对相同算法。最后对齐并核对已经保存的 fresh-162 距离矩阵，防止误读旧 FC 距离。

In [ ]:
ids = neural_unit.index
chemical_values = squareform(pdist(chemical_log2.to_numpy(), metric='euclidean') / np.sqrt(162))
neural_values = np.clip(squareform(pdist(neural_unit.to_numpy(), metric='cosine')), 0, 2)
chemical_distance = pd.DataFrame(chemical_values, index=ids, columns=ids)
neural_distance = pd.DataFrame(neural_values, index=ids, columns=ids)

function_chemical, function_neural = distance_matrices(chemical_log2, neural_unit)
saved_chemical = pd.read_csv(DIST / 'tables/chemical_distance_matrix.csv', index_col=0).loc[ids, ids]
saved_neural = pd.read_csv(DIST / 'tables/neural_distance_matrix.csv', index_col=0).loc[ids, ids]
for actual, reference in [(chemical_distance, function_chemical), (neural_distance, function_neural),
                          (chemical_distance, saved_chemical), (neural_distance, saved_neural)]:
    np.testing.assert_allclose(actual, reference, atol=1e-12, rtol=1e-12)
pd.Series({'Maximum chemical difference from saved matrix': np.max(np.abs(chemical_distance-saved_chemical)),
           'Maximum neural difference from saved matrix': np.max(np.abs(neural_distance-saved_neural))})
np.testing.assert_allclose(neural_distance, main_neural_distance.loc[ids, ids], atol=1e-12, rtol=1e-12)

## 对照 4. 上三角配对、描述性关系与 162 项图

先明确哪些值进入关系图：只取对角线以上，不重复计算对称项，不把自身距离算进去。每个配对共享原来的菌株，因此 5,565 对不是 5,565 个独立生物重复。这里只报告描述性相关，不计算普通独立样本的 p 值。

In [ ]:
i, j = np.triu_indices(len(ids), k=1)
pairs = pd.DataFrame({'strain_a': ids[i], 'strain_b': ids[j],
                      'chemical_distance': chemical_values[i, j],
                      'neural_distance': neural_values[i, j]})
pairs['same_genus'] = context.loc[pairs.strain_a, 'genus'].to_numpy() == context.loc[pairs.strain_b, 'genus'].to_numpy()
assert len(pairs) == 5565 and not pairs[['strain_a','strain_b']].duplicated().any()
x, y = pairs.chemical_distance.to_numpy(), pairs.neural_distance.to_numpy()
pearson_r = np.corrcoef(x, y)[0, 1]
spearman_rho = np.corrcoef(rankdata(x), rankdata(y))[0, 1]
metrics = describe_correspondence(paired_distances(chemical_distance, neural_distance, context))
np.testing.assert_allclose([pearson_r, spearman_rho], [metrics['pearson_r'], metrics['spearman_rho']])
display(pd.Series(metrics, name='Current 162-annotation input'))

### 162 项对照图的色标

仅这张对照图的神经距离热图使用可调的线性范围 `NEURAL_HEATMAP_LIMITS = (0, 1.2)`，让占多数的较小距离能够展开显示，0.6 对应色图的白色中点。72/5,565 个唯一菌株对（1.3%）大于 1.2，颜色饱和为最深红；色条顶端三角表示范围外仍有数值。红色表示显示范围内的较大距离，不等同于负 cosine。

这只改变颜色映射，完整距离、相关指标、样本顺序及右侧配对图仍使用原值。化学热图保持原来的色标。

In [ ]:
# Use the atlas ordering on both matrices; chemistry does not reorder either plot.
order = pd.read_csv(RESP / 'tables/figure_row_order.csv').sort_values('position').strain.tolist()
assert set(order) == set(ids) and len(order) == len(ids)
# Heatmap display only: change this tuple to adjust contrast.
NEURAL_HEATMAP_LIMITS = (0.0, 1.2)
fig = plot_global_comparison(
    chemical_distance, neural_distance, order=order,
    neural_limits=NEURAL_HEATMAP_LIMITS,
)
if SAVE_FIGURES:
    save_figure(fig, figure_dir('supporting') / 'fig03_global_chemical_neural_162concentration')
plt.show()

pairs.to_csv(table_dir() / 'fig03_162concentration_matched_pairs.csv', index=False)
pd.Series(metrics).to_json(table_dir() / 'fig03_162concentration_descriptive_correspondence.json', indent=2)

# Record saturation using unique off-diagonal pairs, not duplicated matrix cells.
display_parameters = {
    'role': 'supporting_162_concentration',
    'chemical_input': '162 complete positive QC-passed log2 concentration annotations',
    'neural_heatmap_limits': list(NEURAL_HEATMAP_LIMITS),
    'neural_heatmap_normalization': 'linear',
    'n_unique_pairs': len(pairs),
    'n_pairs_below_display_min': int((pairs.neural_distance < NEURAL_HEATMAP_LIMITS[0]).sum()),
    'n_pairs_above_display_max': int((pairs.neural_distance > NEURAL_HEATMAP_LIMITS[1]).sum()),
    'colorbar_extensions': 'Mark any values outside the displayed color range',
    'distance_values_modified': False,
    'pair_plot_y_limits': [0, 2],
}
(table_dir() / 'fig03_162concentration_display_parameters.json').write_text(json.dumps(display_parameters, indent=2) + '\n')


**162 项对照的读图：**两张 RDM 用相同菌株顺序，但色阶单位不同；相同颜色不代表相同绝对距离。右图反映这两种已定义距离之间的总体对应。无论关联大小如何，都不能据此推断机制、化学等价，或“神经测量优于 LC–MS”。这一步给出全样本背景，下一本 notebook 再讨论具体局部关系。

## 对照 5. 162 项探索：整体配对中，属内与属间各占多少？

下面直接展示两类配对的数量和距离分布。这是理解总体散点组成的辅助图；大量菌株对存在依赖，直方图描述分布形状，不给出独立重复的置信区间。

In [ ]:
display(pairs.groupby('same_genus').agg(n_pairs=('strain_a','size'),
    chemical_median=('chemical_distance','median'), neural_median=('neural_distance','median')))
fig, axes = plt.subplots(1,2,figsize=(10,3.4),layout='constrained')
for ax, column, xlabel in zip(axes, ['chemical_distance','neural_distance'],
                            ['Chemical RMS log2 difference','Neural 1 − cosine']):
    bins = np.linspace(0, pairs[column].max(), 26)
    for same, label, color in [(True,'Within genus','#367FA1'),(False,'Between genera','#C96F3E')]:
        values = pairs.loc[pairs.same_genus == same, column]
        ax.hist(values, bins=bins, density=True, histtype='step', lw=1.8, color=color, label=label)
    ax.set(xlabel=xlabel, ylabel='Density'); ax.legend(frameon=False,fontsize=9)
if SAVE_FIGURES:
    save_figure(fig, figure_dir('supporting') / 'fig03_pooled_within_between')
plt.show()

## 对照 6. 162 项探索：逐属比较，以及化学/神经各自的属均值结构

上面的总体直方图会让株数多的属贡献更多配对。既有逐属分析则对每个目标属的各个外属等权；下面重画其保存结果。横线仅连接两个中位数，不是误差条。

随后分别展示两个模态已经独立得到的属均值结果：化学共变模块与神经组合。两者分别定义和排序；这些图提供局部分析的背景，不等于已证明跨模态对应。

In [ ]:
genus_summary = pd.read_csv(DIST / 'tables/genus_summary.csv')
fig = plot_within_between(genus_summary)
if SAVE_FIGURES:
    save_figure(fig, figure_dir('supporting') / 'fig03_genus_balanced_distances')
plt.show()
display(genus_summary[['genus','n_strains','chemical_median_ratio','neural_median_ratio']])

In [ ]:
display(Image(filename=str(PATTERNS / 'chemical/figures/01_chemical_module_centers.png'), width=1000))
display(Image(filename=str(PATTERNS / 'neural/figures/01_neural_genus_centered_profiles.png'), width=1000))

## 支持探索：已有 HMDS 与历史样本对

下面的 **106 株 HMDS 三视图**使用 380 项 log₂FC，与恢复后的全局主图采用同一套化学表示；它仍是支持图，点色来自对应的 chemical PCo1。这里直接查看已验证的 embedding，不重新拟合。

A231/A232 整张旧例图使用早期化学输入和七神经元系数，继续作为历史探索记录。它不能与当前 13 神经元主图的坐标或数值混用。

In [ ]:
if SHOW_HISTORICAL:
    display(Markdown('**Supporting HMDS: the same 380-feature reference-based chemistry as the restored main Fig03.**'))
    display(Image(filename=str(RESP / 'figures/03c_neural_chemical_hmds_3d.png'), width=1100))
    display(Markdown('**Historical pair illustration: legacy chemistry and seven-neuron coefficients.**'))
    display(Image(filename=str(report_path('pair_example') / 'sample_comparison_draft.png'), width=1100))

## 导出与来源

主图、对照图及计算表只写入 `poster/`。普通 `fig03_matched_pairs`、`fig03_descriptive_correspondence`、`fig03_display_parameters`、`fig03_sources` 对应 **380 项主图**；`fig03_162concentration_*` 明确对应 **162 项对照**。

原报告保持原状。本 notebook 可以从干净 kernel 顺序执行，不依赖 Figure 1/2 先运行，也不调用旧目录中的全流程脚本。

In [ ]:
source_paths = [CHEM / 'tables' / name for name in [
    'fresh_chemical_log2.csv','neural_unit_coefficients.csv','sample_context.csv','fresh_feature_metadata.csv']]
source_paths += [DIST / 'tables' / name for name in [
    'chemical_distance_matrix.csv','neural_distance_matrix.csv','genus_summary.csv']]
source_paths += [RESP / 'tables/figure_row_order.csv', ROOT / 'poster/src/poster_analysis/global_comparison.py']
provenance = pd.DataFrame([{'path': str(p.relative_to(ROOT)),
    'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in source_paths])
provenance.to_csv(table_dir() / 'fig03_162concentration_sources.csv', index=False)
display(provenance[['path']])
comparison_summary = pd.DataFrame([
    {'role': 'Main: 380 reference-based log2FC', **main_metrics},
    {'role': 'Supporting: 162 log2 concentrations', **metrics},
])
comparison_summary.to_csv(table_dir() / 'fig03_representation_comparison.csv', index=False)
display(comparison_summary)
display(Markdown(f'主图采用 **380 项 FC**：Pearson r = **{main_metrics["pearson_r"]:.3f}**，'
                f'Spearman rho = **{main_metrics["spearman_rho"]:.3f}**；'
                f'162 项浓度对照：r = **{metrics["pearson_r"]:.3f}**，rho = **{metrics["spearman_rho"]:.3f}**。'))

下一步阅读 [Figure 4：局部化学状态](04_local_chemical_states.ipynb)。在那里，问题从“整个化学谱的距离是否相似”收敛为“一个属内的特定化学共变状态对应怎样的神经组合”。不再重复本 notebook 的全体样本关系。